In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.default.olist_orders_dataset")

silver_orders = (
    orders
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
)

(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_orders")
)

print("Pedidos na Silver:", silver_orders.count())

In [0]:
from pyspark.sql import functions as F

tables = {
    "olist_customers_dataset": {
        "target": "silver_customers",
        "keys": ["customer_id"],
    },
    "olist_order_items_dataset": {
        "target": "silver_order_items",
        "keys": ["order_id", "order_item_id"],
    },
    "olist_order_payments_dataset": {
        "target": "silver_order_payments",
        "keys": ["order_id", "payment_sequential"],
    },
    "olist_order_reviews_dataset": {
        "target": "silver_order_reviews",
        "keys": ["review_id", "order_id"],
    },
    "olist_products_dataset": {
        "target": "silver_products",
        "keys": ["product_id"],
    },
    "olist_sellers_dataset": {
        "target": "silver_sellers",
        "keys": ["seller_id"],
    },
    "olist_geolocation_dataset": {
        "target": "silver_geolocation",
        "keys": [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
        ],
    },
    "product_category_name_translation": {
        "target": "silver_category_translation",
        "keys": ["product_category_name"],
    },
}

for source, config in tables.items():
    df = spark.table(f"workspace.default.{source}")

    # Remover registros sem as chaves necessárias
    valid_keys = F.lit(True)
    for key in config["keys"]:
        valid_keys = valid_keys & F.col(key).isNotNull()

    df = df.filter(valid_keys).dropDuplicates(config["keys"])

    target = f"workspace.default.{config['target']}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target)
    )

    print(f"{config['target']}: {df.count()} registros")

silver_customers: 99441 registros
silver_order_items: 112650 registros
silver_order_payments: 103886 registros
silver_order_reviews: 101889 registros
silver_products: 32951 registros
silver_sellers: 3095 registros
silver_geolocation: 720154 registros
silver_category_translation: 71 registros


In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.default.silver_orders")
customers = spark.table("workspace.default.silver_customers")
items = spark.table("workspace.default.silver_order_items")

orders_without_customer = (
    orders.alias("o")
    .join(
        customers.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left_anti"
    )
    .count()
)

items_without_order = (
    items.alias("i")
    .join(
        orders.alias("o"),
        F.col("i.order_id") == F.col("o.order_id"),
        "left_anti"
    )
    .count()
)

print("Pedidos sem cliente:", orders_without_customer)
print("Itens sem pedido:", items_without_order)

if orders_without_customer > 0 or items_without_order > 0:
    raise ValueError("Falha na validação de integridade da camada Silver.")

print("Validações da Silver concluídas com sucesso.")

Pedidos sem cliente correspondente: 0
Itens sem pedido correspondente: 0
